In [1]:
import numpy as np
import math
from pprint import pprint
from collections import Counter, defaultdict
from itertools import combinations
from collections import deque
from itertools import permutations, product
from tqdm import tqdm

from board_check import board_check, combo_counter

In [2]:
d2_l1 = np.array([[1, 1, 1], [1, 0, 0], [1, 0, 0]])

d2_l3 = np.flip(np.flip(d2_l1, axis=0), axis=1) #axis 0: flip rows, 1: flip cols
print(d2_l1)
print(d2_l3)

[[1 1 1]
 [1 0 0]
 [1 0 0]]
[[0 0 1]
 [0 0 1]
 [1 1 1]]


In [3]:
def xnor_2d(a, b):
    c = np.zeros(a.shape, dtype=int)
    for i in range(c.shape[0]):
        for j in range(c.shape[1]): #gross
            a_val = a[i][j] ; b_val = b[i][j]
            if {a_val, b_val} == {1, 0}:
                c[i][j] = 0
            else:
                c[i][j] = 1
    return c

d2_l2 = xnor_2d(d2_l1, d2_l3)
d2_l2

array([[0, 0, 1],
       [0, 1, 0],
       [1, 0, 0]])

In [4]:
final_2d = np.array([d2_l1, d2_l2, d2_l3])
final_2d

array([[[1, 1, 1],
        [1, 0, 0],
        [1, 0, 0]],

       [[0, 0, 1],
        [0, 1, 0],
        [1, 0, 0]],

       [[0, 0, 1],
        [0, 0, 1],
        [1, 1, 1]]])

In [5]:
def d3_cards(a):
    cards = []
    for i in range(3):
        for j in range(3):
            for k in range(3):
                if a[i][j][k] == 0:
                    cards.append((i, j, k))
    return cards

def d3_sets(cards):
    sets = []
    for a, b, c in combinations(cards, 3):
        good = True
        for i in range(3):
            if len({a[i], b[i], c[i]}) == 2:
                good = False
                break
        if good:
            sets.append([a, b, c])
    print('sets')
    return sets
pprint(d3_sets(d3_cards(final_2d)))

sets
[[(0, 1, 1), (1, 1, 2), (2, 1, 0)],
 [(0, 1, 1), (1, 2, 1), (2, 0, 1)],
 [(0, 1, 1), (1, 2, 2), (2, 0, 0)],
 [(0, 1, 2), (1, 1, 0), (2, 1, 1)],
 [(0, 1, 2), (1, 2, 1), (2, 0, 0)],
 [(0, 2, 1), (1, 0, 1), (2, 1, 1)],
 [(0, 2, 1), (1, 1, 2), (2, 0, 0)],
 [(0, 2, 2), (1, 0, 0), (2, 1, 1)],
 [(0, 2, 2), (1, 0, 1), (2, 1, 0)],
 [(0, 2, 2), (1, 1, 0), (2, 0, 1)],
 [(1, 0, 0), (1, 1, 2), (1, 2, 1)],
 [(1, 0, 1), (1, 1, 0), (1, 2, 2)]]


In [2]:
def find_third(a, b):
    c = []
    for i in range(3):
        if a[i] == b[i]:
            c.append(a[i])
        else: 
            c.append([j for j in (0, 1, 2) if j not in (a[i], b[i])][0])
    return tuple(c)

def all_symmetries(coords, n=3):
    off = (n - 1) / 2

    def transform(p, perm, signs):
        c = [p[k] - off for k in range(3)]
        c = [signs[k] * c[perm[k]] for k in range(3)]
        return tuple(int(round(v + off)) for v in c)

    results = set()
    for perm in permutations(range(3)):
        for signs in product((1, -1), repeat=3):
            board = frozenset(transform(p, perm, signs) for p in coords)
            results.add(board)
    return results

def min_rotation(board):
    return min(tuple(sorted(b)) for b in all_symmetries(board))



In [ ]:
all_count = defaultdict(set)

all_cards = []
for one in range(3):
    for two in range(3):
        for three in range(3):
            all_cards.append((one, two, three))

for start_one, start_two in tqdm(combinations(all_cards, 2), total=combo_counter(81, 2)): #DEDUP ROTATIONS!!
    start = {start_one, start_two}
    seen = {min_rotation(start)}
    q = deque([start])
    ends = []

    longest = 2

    while True:
        if len(q) == 0:
            break
        
        trial_board = q.popleft()

        if len(trial_board) > longest:
            longest = len(trial_board)
            # print(longest, len(q))

        cant = [find_third(a, b) for a, b in combinations(trial_board, 2)]
        nexts = [card for card in all_cards if card not in trial_board and card not in cant]
        if not nexts:
            ends.append(trial_board)
        else:
            for n in nexts: 
                canon = min_rotation(trial_board | {n})
                if canon not in seen:
                    q.append(set(canon))
                    seen.add(canon)

    print(f'ends by len: {Counter([len(x) for x in ends], reverse=True)}')
    for end in ends:
        all_count[len(end)].add(frozenset(end))
pprint(all_count)

  0%|          | 0/3240 [00:00<?, ?it/s]

  0%|          | 0/3240 [00:33<?, ?it/s]

ends by len: Counter({8: 901, 9: 55, 'reverse': 1})


TypeError: cannot use 'set' as a set element (unhashable type: 'set')

In [6]:
longest = max(ends, key=len)
print(len(longest), longest)
sets = board_check(list(longest))
print(f'sets: {sets}')

9 {(1, 0, 1), (0, 1, 0), (0, 0, 0), (1, 1, 2), (1, 0, 0), (1, 2, 2), (2, 1, 2), (0, 0, 1), (0, 1, 1)}
sets: (0, [])


# 4D version


In [ ]:
d3_l1 = final_2d
d3_l3 = np.flip(final_2d, axis=1)

In [ ]:
def xnor_3d(a, b):
    c = np.zeros(a.shape, dtype=int)
    for i in range(c.shape[0]):
        for j in range(c.shape[1]): #gross
            for k in range(c.shape[2]):
                a_val = a[i][j][k] ; b_val = b[i][j][k]
                if {a_val, b_val} == {1, 0}:
                    c[i][j][k] = 1
                else:
                    c[i][j][k] = 1
    return c

d3_l2 =  xnor_3d(d3_l1, d3_l3)
# d3_l2 = np.array([1, 1, 1])
d3_l2

array([[[1, 1, 1],
        [1, 1, 1],
        [1, 1, 1]],

       [[1, 1, 1],
        [1, 1, 1],
        [1, 1, 1]],

       [[1, 1, 1],
        [1, 1, 1],
        [1, 1, 1]]])

In [ ]:
final_3d = np.array([d3_l1, d3_l2, d3_l3])
print(final_3d.shape, math.prod(final_3d.shape))
pprint(final_3d)

(3, 3, 3, 3) 81
array([[[[1, 0, 0],
         [0, 1, 0],
         [0, 0, 1]],

        [[0, 1, 0],
         [1, 1, 1],
         [0, 1, 0]],

        [[0, 0, 1],
         [0, 1, 0],
         [1, 0, 0]]],


       [[[1, 1, 1],
         [1, 1, 1],
         [1, 1, 1]],

        [[1, 1, 1],
         [1, 1, 1],
         [1, 1, 1]],

        [[1, 1, 1],
         [1, 1, 1],
         [1, 1, 1]]],


       [[[0, 0, 1],
         [0, 1, 0],
         [1, 0, 0]],

        [[0, 1, 0],
         [1, 1, 1],
         [0, 1, 0]],

        [[1, 0, 0],
         [0, 1, 0],
         [0, 0, 1]]]])


In [ ]:
def get_cards(a):
    board = []
    for one in range(3):
        for two in range(3):
            for three in range(3):
                for four in range(3):
                    if a[one][two][three][four] == 0:
                        board.append((one, two, three, four))
    return board

board = get_cards(final_3d)
print(len(board))
pprint(board)

32
[(0, 0, 0, 1),
 (0, 0, 0, 2),
 (0, 0, 1, 0),
 (0, 0, 1, 2),
 (0, 0, 2, 0),
 (0, 0, 2, 1),
 (0, 1, 0, 0),
 (0, 1, 0, 2),
 (0, 1, 2, 0),
 (0, 1, 2, 2),
 (0, 2, 0, 0),
 (0, 2, 0, 1),
 (0, 2, 1, 0),
 (0, 2, 1, 2),
 (0, 2, 2, 1),
 (0, 2, 2, 2),
 (2, 0, 0, 0),
 (2, 0, 0, 1),
 (2, 0, 1, 0),
 (2, 0, 1, 2),
 (2, 0, 2, 1),
 (2, 0, 2, 2),
 (2, 1, 0, 0),
 (2, 1, 0, 2),
 (2, 1, 2, 0),
 (2, 1, 2, 2),
 (2, 2, 0, 1),
 (2, 2, 0, 2),
 (2, 2, 1, 0),
 (2, 2, 1, 2),
 (2, 2, 2, 0),
 (2, 2, 2, 1)]


In [ ]:
sets, set_list = board_check(board)
print(sets)
pprint(set_list)

40
[[(0, 0, 0, 1), (0, 0, 1, 2), (0, 0, 2, 0)],
 [(0, 0, 0, 1), (0, 1, 0, 2), (0, 2, 0, 0)],
 [(0, 0, 0, 1), (0, 1, 2, 0), (0, 2, 1, 2)],
 [(0, 0, 0, 1), (0, 1, 2, 2), (0, 2, 1, 0)],
 [(0, 0, 0, 2), (0, 0, 1, 0), (0, 0, 2, 1)],
 [(0, 0, 0, 2), (0, 1, 0, 0), (0, 2, 0, 1)],
 [(0, 0, 0, 2), (0, 1, 2, 2), (0, 2, 1, 2)],
 [(0, 0, 1, 0), (0, 1, 0, 2), (0, 2, 2, 1)],
 [(0, 0, 1, 0), (0, 1, 2, 0), (0, 2, 0, 0)],
 [(0, 0, 1, 0), (0, 1, 2, 2), (0, 2, 0, 1)],
 [(0, 0, 1, 2), (0, 1, 0, 0), (0, 2, 2, 1)],
 [(0, 0, 1, 2), (0, 1, 0, 2), (0, 2, 2, 2)],
 [(0, 0, 1, 2), (0, 1, 2, 0), (0, 2, 0, 1)],
 [(0, 0, 2, 0), (0, 1, 0, 0), (0, 2, 1, 0)],
 [(0, 0, 2, 0), (0, 1, 2, 2), (0, 2, 2, 1)],
 [(0, 0, 2, 1), (0, 1, 0, 0), (0, 2, 1, 2)],
 [(0, 0, 2, 1), (0, 1, 0, 2), (0, 2, 1, 0)],
 [(0, 0, 2, 1), (0, 1, 2, 0), (0, 2, 2, 2)],
 [(0, 2, 0, 0), (0, 2, 1, 2), (0, 2, 2, 1)],
 [(0, 2, 0, 1), (0, 2, 1, 0), (0, 2, 2, 2)],
 [(2, 0, 0, 0), (2, 0, 1, 2), (2, 0, 2, 1)],
 [(2, 0, 0, 0), (2, 1, 0, 2), (2, 2, 0, 1)],
 [(2, 0

In [ ]:
pprint([sorted(s) for s in set_list if (0, 0, 0, 1) in s])

[[(0, 0, 0, 1), (0, 0, 1, 2), (0, 0, 2, 0)],
 [(0, 0, 0, 1), (0, 1, 0, 2), (0, 2, 0, 0)],
 [(0, 0, 0, 1), (0, 1, 2, 0), (0, 2, 1, 2)],
 [(0, 0, 0, 1), (0, 1, 2, 2), (0, 2, 1, 0)]]


In [ ]:
set_counts = Counter([x for s in set_list for x in s])
set_counts

Counter({(0, 0, 0, 1): 4,
         (0, 0, 1, 2): 4,
         (0, 1, 0, 2): 4,
         (0, 1, 2, 0): 4,
         (0, 2, 1, 2): 4,
         (0, 1, 2, 2): 4,
         (0, 2, 1, 0): 4,
         (0, 0, 1, 0): 4,
         (0, 0, 2, 1): 4,
         (0, 1, 0, 0): 4,
         (0, 2, 0, 1): 4,
         (0, 2, 2, 1): 4,
         (2, 0, 1, 2): 4,
         (2, 0, 2, 1): 4,
         (2, 1, 0, 2): 4,
         (2, 2, 0, 1): 4,
         (2, 1, 2, 0): 4,
         (2, 2, 1, 0): 4,
         (2, 0, 0, 1): 4,
         (2, 0, 1, 0): 4,
         (2, 1, 0, 0): 4,
         (2, 2, 1, 2): 4,
         (2, 1, 2, 2): 4,
         (2, 2, 2, 1): 4,
         (0, 0, 2, 0): 3,
         (0, 2, 0, 0): 3,
         (0, 0, 0, 2): 3,
         (0, 2, 2, 2): 3,
         (2, 0, 0, 0): 3,
         (2, 0, 2, 2): 3,
         (2, 2, 0, 2): 3,
         (2, 2, 2, 0): 3})